## Message Usage

In [35]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

## Message Object List Demo

In [5]:
# 从.env文件中加载环境变量
load_dotenv(override=True)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")
model = init_chat_model(
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
    model_provider="openai",
    model="openai/gpt-5.6-luna"
)
messages = [
    SystemMessage(
        "你是一个信息抽取器。你会收到多条来自不同发言者的 user 消息。每条消息 可能带有 name 字段。你的任务是：严格根据每条消息的 name 提取发言者及其观点，并输出 JSON。禁止使用“第一个人 / "
        "第二个人”这种相对称呼。若某条消息没有 name，则输出 unknown。输出 格式：{\"speakers\":[{\"name\":\"...\",\"claim\":\"...\"}]}"),
    HumanMessage(
        content="我认为 1+1=2",
        name="Bob"
    ),
    HumanMessage(
        content="我认为 1+1>2",
        name="Tom"
    ),
    HumanMessage(
        content="请列出谁说了什么，不要判断对错。",
        name="audience"
    )
]
response = model.invoke(messages)
print(response.content)

{"speakers":[{"name":"unknown","claim":"我认为 1+1=2"},{"name":"unknown","claim":"我认为 1+1>2"},{"name":"unknown","claim":"请列出谁说了什么，不要判断对错。"}]}


## ToolMessage参数列表

In [32]:
model = init_chat_model(
    model_provider="openrouter",
    model="openai/gpt-5.6-luna"
)


def get_weather(city: str) -> str:
    return "不错哦~"


# 模拟模型绑定工具
model_with_tools = model.bind_tools([get_weather])

user_message = HumanMessage(content="北京天气如何")

ai_message = {
    "role": "assistant",
    "content": "",
    "tool_calls": [{
        "name": "get_weather",
        "args": {"location": "北京"},
        "id": "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
    }]
}
tool_message = {
    "role": "tool",
    "content": "今天北京天气晴朗，万里无云~",
    "tool_call_id": "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
}
messages = [
    user_message,
    ai_message,
    tool_message
]
response = model.invoke(messages)
print(response)

content='今天北京天气晴朗，万里无云。' additional_kwargs={} response_metadata={'model_name': 'openai/gpt-5.6-luna', 'id': 'gen-1784448210-WljrN4mReWzZMdDaRaRD', 'created': 1784448210, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter'} id='lc_run--019f7966-d59b-77c3-b32b-6c2eecdf3d17-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 49, 'output_tokens': 15, 'total_tokens': 64, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}}


In [39]:
ai_message = AIMessage(content=[],
                       tool_calls=[{
                           "name": "get_weather",
                           "args": {"city": "Shanghai"},
                           "id": "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
                       }])
tool_message = ToolMessage(content="今天北京天气晴朗",
                           tool_call_id="call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s")
messages = [
    user_message,
    ai_message,
    tool_message
]
response = model.invoke(messages)
print(response)

content='今天北京天气晴朗。' additional_kwargs={'reasoning_content': "**Clarifying weather details**\n\nI see there’s a bit of confusion here. The tool indicates it’s sunny, but it seems the location is Shanghai instead of Beijing, which the user asked for. I should mention the sunny weather, although it might not include a temperature reading. It's essential to provide the right location and details to be helpful. I'll confirm what I can about the weather without mixing up the cities!", 'reasoning_details': [{'type': 'reasoning.summary', 'summary': "**Clarifying weather details**\n\nI see there’s a bit of confusion here. The tool indicates it’s sunny, but it seems the location is Shanghai instead of Beijing, which the user asked for. I should mention the sunny weather, although it might not include a temperature reading. It's essential to provide the right location and details to be helpful. I'll confirm what I can about the weather without mixing up the cities!", 'format': 'openai-responses-v

## 对话历史优化

In [48]:
def keep_recent_messages(conversation: list[dict], max_pairs: int = 3):
    system_prompt = [message for message in conversation if message.get("role") == "system"]
    user_prompt = [message for message in conversation if message.get("role") != "system"]

    optimized_prompt = user_prompt[-max_pairs * 2:]
    return system_prompt + optimized_prompt


In [49]:
# 初始化
long_conversation = [
    {"role": "system", "content": "你是 Python 导师"}
]
# 第 1 轮
long_conversation.append({"role": "user", "content": "什么是列表？用一句解释"})
r1 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r1.content})
# 第 2 轮
long_conversation.append({"role": "user", "content": "列表和元组有什么区别？用一句解释"})
r2 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r2.content})
# 第 3 轮
long_conversation.append({"role": "user", "content": "什么是字典呢？用一句解释"})
r3 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r3.content})
print(f"原始消息数: {len(long_conversation)}")
# 优化：只保留最近 2 轮
optimized = keep_recent_messages(long_conversation, max_pairs=2)
print(f"优化后消息数: {len(optimized)}")
print(f"保留的内容: system + 最近2轮对话")
# 添加新的用户问题
optimized.append({"role": "user", "content": "我第一个问题问的是什么？"})
# 使用优化后的历史
response = model.invoke(optimized)
print(f"\nAI 回复: {response.content}")

原始消息数: 7
优化后消息数: 5
保留的内容: system + 最近2轮对话

AI 回复: 你第一个问题问的是：“列表和元组有什么区别？用一句解释”。


## Chatbox

In [51]:
MAX_PAIRS_HISTORY = 3
EXIT_WORD_LIST = ["quit", "QUIT", "exit", "EXIT"]

model = init_chat_model(model="deepseek:deepseek-v4-flash")

system_prompt = {"role": "system",
                 "content": "You are a Chatbox, response user in Chinese"}
conversations = [system_prompt]
print("-" * 10, f"Chatbox, enter {EXIT_WORD_LIST} to exit.", "-" * 10)
loop: int = 1
while True:
    print("-" * 10, f"The {loop} loop start", "-" * 10)
    user_prompt = input("Input your prompt...")
    if user_prompt in EXIT_WORD_LIST:
        print("Exiting...")
        break
    print(f"User: {user_prompt}")
    keep_recent_messages(conversations, max_pairs=MAX_PAIRS_HISTORY)
    conversations.append({"role": "user", "content": user_prompt})

    current_response = ""
    print("LLM:", end=" ")
    for chunk in model.stream(input=conversations):
        if chunk.content:
            print(chunk.content, end="", flush=True)
            current_response += chunk.content

    conversations.append({"role": "assistant", "content": current_response})

    print("-" * 10, f"\nThe {loop} loop end", "-" * 10, end="\n\n")
    loop += 1


---------- Chatbox, enter ['quit', 'QUIT', 'exit', 'EXIT'] to exit. ----------
---------- The 1 loop start: ----------
User: Hello
LLM: 你好！有什么我可以帮你的吗？---------- The 1 loop end ----------

---------- The 2 loop start: ----------
Exiting...
